In [ ]:
from pathlib import Path
import math
import torch
from torch import nn

checkpoint = Path("./checkpoints/mnist_cnn_best.pth")
if not checkpoint.is_file():
    raise FileNotFoundError(
        f"找不到 {checkpoint.resolve()}；请确认 Jupyter 当前目录是项目目录"
    )

# 与训练时完全相同的网络结构
model = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(4, 8, kernel_size=3, stride=1, padding=0),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(200, 10),
)

state = torch.load(checkpoint, map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.eval()

# 用前向钩子记录真实的逐层输入、输出形状
records = []
handles = []

for name, layer in model.named_children():
    def record(module, inputs, output, name=name):
        records.append({
            "name": name,
            "module": module,
            "input": tuple(inputs[0].shape),
            "output": tuple(output.shape),
        })

    handles.append(layer.register_forward_hook(record))

with torch.inference_mode():
    final_output = model(torch.zeros(1, 1, 28, 28))

for handle in handles:
    handle.remove()

def shape_text(shape):
    return "×".join(map(str, shape))

def size_text(value):
    if isinstance(value, tuple):
        return shape_text(value)
    return str(value)

lines = [
    "# MNIST CNN 网络规格",
    "",
    "来源：`./checkpoints/mnist_cnn_best.pth`；输入为单张 `1×28×28` 图像。",
    "",
    "| 编号 | 层类型 | 输入形状 | 输出形状 | 权重形状 | 偏置形状 | Kernel | Stride | Padding | 激活/作用 | 参数量 | MAC | 3×3 PE |",
    "|---|---|---|---|---|---|---|---|---|---|---:|---:|---|",
]

total_macs = 0

for item in records:
    name = item["name"]
    layer = item["module"]
    input_shape = item["input"]
    output_shape = item["output"]

    weight = getattr(layer, "weight", None)
    bias = getattr(layer, "bias", None)
    weight_shape = shape_text(weight.shape) if weight is not None else "—"
    bias_shape = shape_text(bias.shape) if bias is not None else "—"
    params = sum(p.numel() for p in layer.parameters(recurse=False))

    kernel = stride = padding = "—"
    role = "—"
    pe = "否"
    macs = 0

    if isinstance(layer, nn.Conv2d):
        kernel = size_text(layer.kernel_size)
        stride = size_text(layer.stride)
        padding = size_text(layer.padding)
        macs = (
            math.prod(output_shape)
            * (layer.in_channels // layer.groups)
            * math.prod(layer.kernel_size)
        )
        pe = "是" if layer.kernel_size == (3, 3) else "否"
    elif isinstance(layer, nn.MaxPool2d):
        kernel = size_text(layer.kernel_size)
        stride = size_text(layer.stride)
        padding = size_text(layer.padding)
        role = "最大池化"
        pe = "否，需池化模块"
    elif isinstance(layer, nn.ReLU):
        role = "ReLU"
        pe = "否，需激活逻辑"
    elif isinstance(layer, nn.Flatten):
        role = "展平"
        pe = "否，需地址映射"
    elif isinstance(layer, nn.Linear):
        macs = math.prod(output_shape) * layer.in_features
        pe = "否，需 FC 计算"

    total_macs += macs

    lines.append(
        f"| {name} | {type(layer).__name__} | "
        f"{shape_text(input_shape)} | {shape_text(output_shape)} | "
        f"{weight_shape} | {bias_shape} | {kernel} | {stride} | "
        f"{padding} | {role} | {params} | {macs} | {pe} |"
    )

weight_count = sum(
    p.numel() for name, p in model.named_parameters()
    if name.endswith(".weight")
)
bias_count = sum(
    p.numel() for name, p in model.named_parameters()
    if name.endswith(".bias")
)
max_feature_elements = max(
    [28 * 28] + [math.prod(item["output"]) for item in records]
)

lines += [
    "",
    f"- 权重：**{weight_count}**",
    f"- 偏置：**{bias_count}**",
    f"- 总参数：**{weight_count + bias_count}**",
    f"- 单张图像总 MAC：**{total_macs}**",
    f"- 最大单个特征图：**{max_feature_elements} 个元素**",
    "- 3×3 PE 负责两层 Conv；ReLU、MaxPool、Flatten 和 FC 另行处理。",
    "- 当前整数参考模型的最终 logits 保留 INT32，不能将整网输出统一假定为 INT8。",
    "",
]

output_path = Path("./network_spec.md")
output_path.write_text("\n".join(lines), encoding="utf-8")

print("Checkpoint loaded:", checkpoint.resolve())
print("Model output shape:", tuple(final_output.shape))
print("Weights:", weight_count)
print("Biases:", bias_count)
print("MACs per image:", total_macs)
print("Largest feature map:", max_feature_elements, "elements")
print("Saved:", output_path.resolve())